In [ ]:
import os
import shutil
import numpy as np
from multiprocessing import Pool
from ReadParticle import read_particle_frames
from frame_renderer import init_worker, render_frame_HSTview


def make_HST_video(particle_file, output_dir, output_name, fps=24, zmout_freq=1000):


    # Read particle and collision data
    Np_seq, time, r_dust, data_p = read_particle_frames(particle_file)

    # Ensure output directory exists for saving frames
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # Precompute axis limits for each frame
    axis_lim_list = []
    cache1 = None
    for i in range(len(time)):
        if i % zmout_freq == 0:
            data_i = data_p[i]
            dust_distance = (data_i[4:, 1]**2 + data_i[4:, 2]**2 + data_i[4:, 3]**2)**0.5
            dis_max = dust_distance.max()
            axis_lim_list.append(dis_max)
            cache1 = dis_max
        else:
            axis_lim_list.append(cache1)

    # Render frames in parallel
    with Pool(initializer=init_worker, initargs=(data_p, time, axis_lim_list, output_dir)) as pool:
        pool.map(render_frame_HSTview, range(len(time)))

    print("All frames rendered. Combining into video...", flush=True)

    # Combine frames into an MP4 using ffmpeg
    os.system(
        f"ffmpeg -y -r {fps} -i {output_dir}/hstview_frame_%04d.png "
        f"-vf 'pad=ceil(iw/2)*2:ceil(ih/2)*2' -vcodec libx264 -crf 18 -pix_fmt yuv420p {output_name}"
    )

    print(f"Animation of HSTview saved as {output_name}", flush=True)


def make_topview_video(particle_file, output_dir, output_name, fps=24, zmout_freq=1000):


    # Read particle and collision data
    Np_seq, time, r_dust, data_p = read_particle_frames(particle_file)

    # Ensure output directory exists for saving frames
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # Precompute axis limits for each frame
    axis_lim_list = [30e3] * num_frames
#     axis_lim_list = []
#     cache1 = None
#     for i in range(len(time)):
#         if i % zmout_freq == 0:
#             data_i = data_p[i]
#             dust_distance = (data_i[4:, 1]**2 + data_i[4:, 2]**2 + data_i[4:, 3]**2)**0.5
#             dis_max = dust_distance.max()
#             axis_lim_list.append(dis_max)
#             cache1 = dis_max
#         else:
#             axis_lim_list.append(cache1)

    # Render frames in parallel
    with Pool(initializer=init_worker, initargs=(data_p, time, axis_lim_list, output_dir)) as pool:
        pool.map(render_frame_topview, range(len(time)))

    print("All frames rendered. Combining into video...", flush=True)

    # Combine frames into an MP4 using ffmpeg
    os.system(
        f"ffmpeg -y -r {fps} -i {output_dir}/topview_frame_%04d.png "
        f"-vf 'pad=ceil(iw/2)*2:ceil(ih/2)*2' -vcodec libx264 -crf 18 -pix_fmt yuv420p {output_name}"
    )

    print(f"Animation of top view saved as {output_name}", flush=True)

    

if __name__ == "__main__":
    # Example usage
    parent_dir = "/home/linfel/linfel_data/data_high_shortterm_run/run_034"
    particle_file = os.path.join(parent_dir, "particles.txt")
    output_dir = os.path.join(parent_dir, "postprocess/HST_frames")
    output_video = os.path.join(parent_dir, "postprocess/HST_view.mp4")
    make_HST_video(particle_file, output_dir, output_video, fps=24)